# Lab | Agent & Vector store — Your turn

**Rebuilt from scratch on a different dataset** (bonus option: a new notebook rather than
edits to the demo).

The demo routes between the *state of the union* speech and the *Ruff* docs. Here I keep the
Ruff tool and replace the speech with **Shakespeare's sonnets**, then rebuild the pipeline —
loader → splitter → vector store → `RetrievalQA` → `Tool` → agent.

**Why the sonnets?** They are about as far from a Python linter as a corpus can get, so a
misrouted question would be obvious. And the two corpora share exactly one interesting word:
Ruff advertises compatibility with the **Black** formatter, and Shakespeare's "dark lady"
sonnets are built on the word *black*. That gives me a real multi-hop question in step 5.

Following the lab's steps:

1. Get the data
2. Rebuild the vector store
3. Rewrite the tool description
4. Rebuild the agent
5. Test it — a sonnets question, a Ruff question, and a multi-hop question
6. Reflect

---

## Setup

`chromadb` and `beautifulsoup4` are installed in the `langchain-v0.2.x` venv already (from
the terminal, not with `!pip install` — an unpinned install from inside the notebook would
upgrade `langchain-core` to 1.x and break the environment).

In [1]:
import os
import warnings

os.environ["USER_AGENT"] = "ironhack-lab-agent-vector-store/1.0"   # silences a WebBaseLoader warning

from dotenv import load_dotenv, find_dotenv

from langchain_community.document_loaders import TextLoader, WebBaseLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_openai import OpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain.agents import AgentType, Tool, initialize_agent

_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("API key loaded:", bool(OPENAI_API_KEY))

API key loaded: True


In [2]:
llm = OpenAI(temperature=0, api_key=OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY)

# One splitter, reused for both corpora.
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

## Steps 1 & 2 — get the data and rebuild the vector store

Downloaded from the repo the lab links to. Note the **raw** URL: the link in the lab
instructions points at GitHub's HTML page, which would give me a web page rather than the
file.

```bash
curl -L -o sonnets.txt "https://raw.githubusercontent.com/martin-gorner/tensorflow-rnn-shakespeare/master/shakespeare/sonnets.txt"
```

Same four moves as any RAG index: **load → split → embed → store**.

In [3]:
sonnet_docs = TextLoader("sonnets.txt").load()
print(f"loaded {len(sonnet_docs)} document, {len(sonnet_docs[0].page_content):,} characters")

sonnet_texts = text_splitter.split_documents(sonnet_docs)
print(f"split into {len(sonnet_texts)} chunks")
print(f"longest chunk: {max(len(t.page_content) for t in sonnet_texts)} characters")

loaded 1 document, 95,662 characters
split into 154 chunks
longest chunk: 814 characters


In [4]:
sonnets_db = Chroma.from_documents(
    sonnet_texts,
    embeddings,
    collection_name="shakespeare-sonnets",   # distinct name, so it can't collide with the Ruff store
)

# Sanity check. Re-running the cell above without restarting the kernel doubles this number:
# Chroma appends rather than replaces, and nothing errors when it does.
print("vectors in collection:", sonnets_db._collection.count())

vectors in collection: 154


In [5]:
sonnets_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=sonnets_db.as_retriever(),
)

# Check retrieval works before wiring it into an agent - debugging a broken
# retriever from inside an agent trace is miserable.
print(sonnets_qa.run("What does Shakespeare say about black in the sonnets?"))

/var/folders/tl/vp52sgtd0n170ds8dv7_2wm00000gn/T/ipykernel_39980/2644551321.py:9: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  print(sonnets_qa.run("What does Shakespeare say about black in the sonnets?"))


 In the sonnets, Shakespeare often uses the color black to represent beauty and love. He also suggests that black is unfairly associated with negative qualities, and that true beauty and love can be found in all colors.


### The second knowledge source — the Ruff FAQ

Kept from the demo, so the agent has something to route *against*. Same pipeline, different
loader: `WebBaseLoader` fetches and strips the HTML instead of reading from disk.

In [6]:
ruff_docs = WebBaseLoader("https://beta.ruff.rs/docs/faq/").load()
print(f"fetched {len(ruff_docs[0].page_content):,} characters")

ruff_texts = text_splitter.split_documents(ruff_docs)
print(f"split into {len(ruff_texts)} chunks")
print(f"longest chunk: {max(len(t.page_content) for t in ruff_texts)} characters  <-- over the 1000 target")

Created a chunk of size 2122, which is longer than the specified 1000


Created a chunk of size 3187, which is longer than the specified 1000


Created a chunk of size 1017, which is longer than the specified 1000


Created a chunk of size 2321, which is longer than the specified 1000


fetched 23,642 characters
split into 23 chunks
longest chunk: 3187 characters  <-- over the 1000 target


The oversized chunks are worth a note. `CharacterTextSplitter` splits on a *single*
separator (`"\n\n"`); when one blank-line-delimited block of scraped HTML is already bigger
than the target, there is no split point inside it and the splitter emits it whole.
`chunk_size` is a target, not a guarantee. The sonnets had no such problem — poems come
pre-separated by blank lines.

In [7]:
ruff_db = Chroma.from_documents(
    ruff_texts,
    embeddings,
    collection_name="ruff-faq",
)
print("vectors in collection:", ruff_db._collection.count())

ruff_qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ruff_db.as_retriever(),
)

vectors in collection: 23


## Step 3 — rewrite the tool descriptions

The agent has never seen either corpus and cannot look inside a vector store. **Everything it
knows about my knowledge bases is the `description` string I write here.** If it routes
badly, the description is the bug.

I wrote each one with three jobs in mind:

1. **When to reach for me** — the topic, in the vocabulary a user would actually use
   ("sonnets", "Shakespeare", "poetry" / "ruff", "linter", "formatter")
2. **What is inside me** — enough detail to tell the two apart
3. **How to call me** — `RetrievalQA` embeds whatever string it receives, so a bare keyword
   retrieves badly and a full question retrieves well

The *"not referencing any obscure pronouns"* clause is copied from the demo's multi-hop cell
and is there for the same reason: on a second hop the agent otherwise tends to write
`Action Input: How does he use it?`, and "it" embeds to nothing useful.

In [8]:
sonnets_tool_description = (
    "useful for when you need to answer questions about Shakespeare's sonnets - their "
    "themes, imagery, language, or the content of a particular sonnet. Use this for any "
    "question about poetry, love, beauty, time, or Shakespeare's verse. "
    "Input should be a fully formed question, not referencing any obscure pronouns "
    "from the conversation before."
)

ruff_tool_description = (
    "useful for when you need to answer questions about ruff, a Python linter and code "
    "formatter written in Rust - its speed, its configuration, and how it compares to "
    "other Python tools such as flake8, isort or Black. "
    "Input should be a fully formed question, not referencing any obscure pronouns "
    "from the conversation before."
)

tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets_qa.run,          # note: .run, not .invoke - a Tool must return a string
        description=sonnets_tool_description,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff_qa.run,
        description=ruff_tool_description,
    ),
]

for t in tools:
    print(f"{t.name}\n  {t.description}\n")

Shakespeare Sonnets QA System
  useful for when you need to answer questions about Shakespeare's sonnets - their themes, imagery, language, or the content of a particular sonnet. Use this for any question about poetry, love, beauty, time, or Shakespeare's verse. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.

Ruff QA System
  useful for when you need to answer questions about ruff, a Python linter and code formatter written in Rust - its speed, its configuration, and how it compares to other Python tools such as flake8, isort or Black. Input should be a fully formed question, not referencing any obscure pronouns from the conversation before.



## Step 4 — rebuild the agent

`max_iterations=3` is a cost control the demo leaves out. The scratchpad accumulates every
previous thought and observation, so step *n* re-sends steps 1..*n*-1 — a wandering agent
gets expensive fast. The default is 15.

In [9]:
agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=3,
)

/var/folders/tl/vp52sgtd0n170ds8dv7_2wm00000gn/T/ipykernel_39980/692813030.py:1: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


## Step 5 — put it to the test

Three questions, as the lab asks. I am reading the `verbose` trace, not just the final
answer — the `Action:` line is the thing under test.

### 5a. A question only the sonnets can answer

In [10]:
agent.invoke(
    "What imagery does Shakespeare use to describe the passing of time in the sonnets?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Shakespeare Sonnets QA System to answer this question.
Action: Shakespeare Sonnets QA System
Action Input: "What imagery does Shakespeare use to describe the passing of time in the sonnets?"


Observation:  Shakespeare uses imagery of waves, clocks, seasons, and natural decay to describe the passing of time in the sonnets.
Thought:

 I now know the final answer.
Final Answer: Shakespeare uses imagery of waves, clocks, seasons, and natural decay to describe the passing of time in the sonnets.

> Finished chain.


{'input': 'What imagery does Shakespeare use to describe the passing of time in the sonnets?',
 'output': 'Shakespeare uses imagery of waves, clocks, seasons, and natural decay to describe the passing of time in the sonnets.'}

### 5b. A question only Ruff can answer

In [11]:
agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 You should consider the differences between ruff and flake8
Action: Ruff QA System
Action Input: "What are the differences between ruff and flake8?"


Observation:  Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.
Thought:

 I now know the differences between ruff and flake8
Final Answer: Ruff has a larger rule set, supports automatic fixing of lint violations, and is written in Rust, while Flake8 supports plugins and custom rules and is written in Python.

> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': 'Ruff has a larger rule set, supports automatic fixing of lint violations, and is written in Rust, while Flake8 supports plugins and custom rules and is written in Python.'}

### 5c. A multi-hop question that needs both tools

> *Which Python code formatter is ruff designed to be compatible with, and how does
> Shakespeare use that same word in his sonnets?*

Neither store can answer this alone. The agent has to query Ruff, learn the answer is
**Black**, and carry that word across into a query against the sonnets — where it lands in
the "dark lady" poems (*"In the old age black was not counted fair"*).

The only thing making the second hop possible is the **scratchpad**: the word "Black" appears
in hop 1's `Observation`, which is still in the prompt when the agent plans hop 2.

In [12]:
agent.invoke(
    "Which Python code formatter is ruff designed to be compatible with, "
    "and how does Shakespeare use that same word in his sonnets?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Ruff QA System to answer this question, as it is specifically designed to answer questions about ruff and its compatibility with other Python tools.
Action: Ruff QA System
Action Input: "What Python code formatter is ruff compatible with?"


Observation:  Ruff is compatible with Black, and can be used as a drop-in replacement for Flake8 and Pylint.
Thought:

 Now I need to use the Shakespeare Sonnets QA System to find out how Shakespeare uses the word "black" in his sonnets.
Action: Shakespeare Sonnets QA System
Action Input: "How does Shakespeare use the word 'black' in his sonnets?"


Observation:  In Shakespeare's sonnets, the word 'black' is often used to describe beauty and love, as well as to contrast with traditional ideas of beauty and societal expectations. It is also used to symbolize mourning and sorrow, as well as the inevitability of time and mortality.
Thought:

 I now know the final answer.
Final Answer: Ruff is compatible with Black, and Shakespeare uses the word 'black' in his sonnets to convey themes of beauty, love, societal expectations, mourning, and the passage of time.

> Finished chain.


{'input': 'Which Python code formatter is ruff designed to be compatible with, and how does Shakespeare use that same word in his sonnets?',
 'output': "Ruff is compatible with Black, and Shakespeare uses the word 'black' in his sonnets to convey themes of beauty, love, societal expectations, mourning, and the passage of time."}

### The same agent as a pure router (`return_direct=True`)

The lab's reflection asks what changes with `return_direct=True`, so here is the comparison.
It tells the executor to stop the moment a tool returns and hand back its output verbatim —
the agent's only remaining job is picking the right drawer.

Same tools, same descriptions, one extra argument.

In [13]:
router_tools = [
    Tool(
        name="Shakespeare Sonnets QA System",
        func=sonnets_qa.run,
        description=sonnets_tool_description,
        return_direct=True,
    ),
    Tool(
        name="Ruff QA System",
        func=ruff_qa.run,
        description=ruff_tool_description,
        return_direct=True,
    ),
]

router_agent = initialize_agent(
    router_tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=3,
)

In [14]:
router_agent.invoke(
    "What imagery does Shakespeare use to describe the passing of time in the sonnets?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Shakespeare Sonnets QA System to answer this question.
Action: Shakespeare Sonnets QA System
Action Input: "What imagery does Shakespeare use to describe the passing of time in the sonnets?"


Observation:  Shakespeare uses imagery of waves, clocks, seasons, and natural elements such as trees and the ocean to describe the passing of time in the sonnets. He also uses imagery of youth and beauty fading with the passage of time.


> Finished chain.


{'input': 'What imagery does Shakespeare use to describe the passing of time in the sonnets?',
 'output': ' Shakespeare uses imagery of waves, clocks, seasons, and natural elements such as trees and the ocean to describe the passing of time in the sonnets. He also uses imagery of youth and beauty fading with the passage of time.'}

In [15]:
router_agent.invoke("Why use ruff over flake8?")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 You should consider the differences between ruff and flake8
Action: Ruff QA System
Action Input: "What are the differences between ruff and flake8?"


Observation:  Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.


> Finished chain.


{'input': 'Why use ruff over flake8?',
 'output': ' Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports plugins and allows for custom and third-party rules. Ruff also has a formatter and can automatically fix its own lint violations, while Flake8 does not have these capabilities. Additionally, Ruff is written in Rust while Flake8 is written in Python.'}

And the multi-hop question, run through the router for comparison:

In [16]:
router_agent.invoke(
    "Which Python code formatter is ruff designed to be compatible with, "
    "and how does Shakespeare use that same word in his sonnets?"
)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


 I should use the Ruff QA System to answer this question, as it is specifically designed to answer questions about ruff and its compatibility with other Python tools.
Action: Ruff QA System
Action Input: "What Python code formatter is ruff compatible with?"


Observation:  Ruff is compatible with Black, and can be used as a drop-in replacement for Flake8 and Pylint.


> Finished chain.


{'input': 'Which Python code formatter is ruff designed to be compatible with, and how does Shakespeare use that same word in his sonnets?',
 'output': ' Ruff is compatible with Black, and can be used as a drop-in replacement for Flake8 and Pylint.'}

## Step 6 — reflection

### Did the agent pick the right tool each time?

Yes — six for six. Reading the `Action:` lines from the traces rather than judging by the
final answers:

| Question | Tool(s) called | Right? |
|---|---|---|
| 5a — Shakespeare's imagery for the passing of time | `Shakespeare Sonnets QA System` | ✅ one call |
| 5b — Why use ruff over flake8? | `Ruff QA System` | ✅ one call |
| 5c — multi-hop (formatter → sonnets) | `Ruff QA System` **then** `Shakespeare Sonnets QA System` | ✅ two calls, right order |
| 5a again, router mode | `Shakespeare Sonnets QA System` | ✅ |
| 5b again, router mode | `Ruff QA System` | ✅ |
| 5c again, router mode | `Ruff QA System` only | ✅ routed right, but see below |

No misrouting, and no wasted calls — each single-source question was answered with exactly
one tool call. That is better than the demo notebook managed on the same model: without
`max_iterations`, its agent made **three** calls on the Ketanji Brown Jackson question,
including a pointless detour into the Ruff store (`Action Input: "Who is Ketanji Brown
Jackson?"` → *"I don't know."*), and queried Ruff three separate times on the flake8
question before settling. Mine did each in one. Same model, same temperature — the
difference is the iteration cap forcing it to commit rather than wander.

I read that as the tool descriptions doing their job. The two corpora are so far apart
semantically that this was never going to be a hard routing problem, so I would not claim
much more than "the descriptions were not the bottleneck here". A harder test would be two
similar corpora.

**The multi-hop question is the result I am most pleased with**, because the second hop
cannot be faked:

```
Action: Ruff QA System
Action Input: "What Python code formatter is ruff compatible with?"
Observation:  Ruff is compatible with Black...

Action: Shakespeare Sonnets QA System
Action Input: "How does Shakespeare use the word 'black' in his sonnets?"
```

The word `black` in the second `Action Input` could only have come from the first
`Observation`. When the question arrived, the agent had no way of knowing it would end up
searching a 16th-century poetry corpus for a colour. The **scratchpad** is the only thing
carrying that fact between steps — and it is also why agents get expensive, since step *n*
re-sends everything from steps 1..*n*-1.

### What happened with `return_direct=True` vs. without it?

Comparing the router runs against the reasoner runs on identical questions:

**1. The trace ends one step earlier.** With `return_direct=True` the output stops at the
`Observation:` — there is no closing `Thought: I now know the final answer` and no
`Final Answer:` line. The executor returns the moment the tool does.

**2. One fewer LLM call per question**, because the agent never reasons over what the tool
returned. Cheaper and faster.

**3. The answer is verbatim from the retrieval chain rather than paraphrased.** Clearest on
5b — the reasoner compressed its observation into a tidier sentence:

> *"Ruff has a larger rule set, supports automatic fixing of lint violations, and is written
> in Rust, while Flake8 supports plugins and custom rules and is written in Python."*

while the router handed back exactly what `ruff_qa.run()` produced, leading space and all:

> *" Ruff has a larger rule set and does not support custom lint rules, while Flake8 supports
> plugins and allows for custom and third-party rules. Ruff also has a formatter and can
> automatically fix its own lint violations, while Flake8 does not have these capabilities.
> Additionally, Ruff is written in Rust while Flake8 is written in Python."*

Same facts, but the router's version is traceable to the source and the reasoner's has been
through a second model pass that dropped the plugin detail. If you care about the answer not
being quietly reworded between the retrieval chain and the user, that matters.

**4. And the real cost — it cannot multi-hop.** Running the *same* multi-hop question through
the router is the clearest evidence in the notebook. It routed correctly to the Ruff tool,
and then simply stopped:

> *"Ruff is compatible with Black, and can be used as a drop-in replacement for Flake8 and
> Pylint."*

Half an answer. It never asked Shakespeare anything, because there is no step 2 — the
executor returned the instant the first tool did. `return_direct=True` and multi-hop
reasoning are mutually exclusive by construction, which is why the demo drops
`return_direct` before its own nbQA question.

**So which to use is a product decision, not a technical one.** Router for FAQ-style lookups
where one source always holds the whole answer: cheaper, faster, verbatim. Reasoner when a
question might span sources or need synthesis — you pay an extra LLM call per question and
accept some paraphrasing in exchange for the agent being able to take a second step.

### One thing I would change

`CharacterTextSplitter(chunk_size=1000)` produced a 3,187-character chunk from the Ruff page
— three times the target — because it splits on a single separator (`"\n\n"`) and one block
of scraped HTML was already that big. The sonnets were fine (longest chunk 814) because poems
come pre-separated by blank lines. For scraped HTML, `RecursiveCharacterTextSplitter` is the
right tool: it falls through `["\n\n", "\n", " ", ""]` until the chunk fits. I would also
set `chunk_overlap` to ~100 rather than 0, so a sentence straddling a boundary isn't cut in
half with neither piece retrievable.